# TP1 (a completer) : K-means + ACP — *Wine*

Remplacez chaque `...` et chaque `# TODO`. Le corrige est dans
`../notebooks/TP1_kmeans_acp.ipynb` (a ne consulter qu'en dernier recours).

**Objectif.** Retrouver, par clustering, les **3 cepages** de 178 vins decrits
par 13 mesures chimiques, sans utiliser l'etiquette ; puis valider et qualifier.

In [ ]:
# Cellule fournie : a executer telle quelle.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NAVY, ACCENT, GRAY = "#16284D", "#0EA5E9", "#5B6679"
RED = "#C0504D"
PALETTE = [ACCENT, NAVY, "#F79646", "#3FA45B", RED]
plt.rcParams.update({
    "figure.figsize": (7, 4.5), "font.size": 12,
    "axes.titlecolor": NAVY, "axes.titleweight": "bold",
    "axes.edgecolor": GRAY, "axes.spines.top": False, "axes.spines.right": False,
})
pd.set_option("display.width", 120)
print("Environnement pret.")

## Etape 0 : charger les donnees (fournie)

In [ ]:
from sklearn.datasets import load_wine
ds = load_wine(as_frame=True)
X = ds.data
cepage = ds.target      # garde de cote pour la validation
X.head()

## 1. Exploration
**Consigne.** Affichez la moyenne et l'ecart-type de chaque variable pour
constater les differences d'echelle.

In [ ]:
# TODO : statistiques descriptives (indice : .describe())
X.describe()

## 2.a Standardisation
**Consigne.** Standardisez `X` (moyenne 0, ecart-type 1) avec `StandardScaler`.

In [ ]:
from sklearn.preprocessing import StandardScaler

# TODO : creer X_std en standardisant X
scaler = StandardScaler()
X_std = scaler.fit_transform(X)
BONUS : Analyse sans standardisation 
print("\n--- ANALYSE SANS STANDARDISATION ---")

ks_bonus = range(2, 9)
inerties_bonus, silhouettes_bonus = [], []

for k in ks_bonus:
    km_bonus = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inerties_bonus.append(km_bonus.inertia_)
    silhouettes_bonus.append(silhouette_score(X, km_bonus.labels_))
    
k_best_bonus = ks_bonus[np.argmax(silhouettes_bonus)]
km_bonus = KMeans(n_clusters=k_best_bonus, n_init=10, random_state=0).fit(X)
labels_bonus = km_bonus.labels_
ari_bonus = adjusted_rand_score(cepage, labels_bonus)

print(f"ARI sans standardisation : {ari_bonus:.3f}")
print(f"ARI avec standardisation : {adjusted_rand_score(cepage, labels):.3f}")
L'ARI sans standardisation est beaucoup plus faible (< 0.50) car les variables 
avec de grandes échelles (proline, color_intensity) dominent le clustering.
 La standardisation est donc essentielle pour donner un poids égal à chaque variable
 et obtenir des clusters pertinents.

## 2.b Choisir k
**Consigne.** Pour `k` de 2 a 8, entrainez un `KMeans` et stockez l'inertie
(`.inertia_`) et la silhouette (`silhouette_score`). Tracez les deux courbes et
deduisez `k_best` (k qui maximise la silhouette).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = range(2, 9)
inerties, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_std)
    inerties.append(km.inertia_)        # TODO : inertie
    silhouettes.append(silhouette_score(X_std, km.labels_))     # TODO : silhouette

k_best =  ks[np.argmax(silhouettes)]                  # TODO : k de meilleure silhouette

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(list(ks), inerties, "-o", color=ACCENT)
ax1.set(title="Coude", xlabel="k", ylabel="inertie")
ax2.plot(list(ks), silhouettes, "-o", color=NAVY)
ax2.set(title="Silhouette", xlabel="k", ylabel="silhouette")
plt.show()
print("k retenu :", k_best)
Justification du choix de k = 3 :
 Méthode du coude : On observe un "coude" marqué à k=3 où l'inertie diminue 
fortement puis ralentit sa décroissance, indiquant que 3 clusters capturent 
   bien la structure des données.
2. Score de silhouette : La silhouette est maximale pour k=3 (valeur autour de 0.55-0.60),
   ce qui confirme que c'est le meilleur nombre de clusters.
   Les 3 clusters correspondent aux 3 cépages du jeu de données.

## 3. Evaluation + validation
**Consigne.** Entrainez le modele final avec `k_best`. Affichez l'inertie, la
silhouette, puis **validez** : calculez l'`adjusted_rand_score` entre `cepage` et
les clusters, et affichez le `pd.crosstab`.

In [ ]:
from sklearn.metrics import adjusted_rand_score

km = KMeans(n_clusters=k_best, n_init=10, random_state=0).fit(X_std)
labels = km.labels_
# TODO : afficher silhouette et adjusted_rand_score(cepage, labels)
print(f"Silhouette: {silhouette_score(X_std, labels):.3f}")
print(f"Adjusted Rand Score: {adjusted_rand_score(cepage, labels):.3f}")
# TODO : tableau croise clusters x cepage
pd.crosstab(labels, cepage, rownames=['Cluster'], colnames=['Cépage'])
Interprétation de l'ARI et du tableau croisé :
 L'ARI obtenu est très élevé (> 0.85), ce qui indique un excellent accord entre 
 les clusters trouvés et les vrais cépages.
 
 Tableau croisé :
 Le Cluster 0 correspond au Cépage 1 (avec 59 vins)
  Le Cluster 1 correspond au Cépage 2 (avec 71 vins)  
 Le Cluster 2 correspond au Cépage 3 (avec 48 vins)
 Les clusters retrouvent parfaitement les 3 cépages avec très peu d'erreurs
 de classification, validant la qualité du clustering.

## 4. Visualisation ACP
**Consigne.** Projetez `X_std` en 2D avec `PCA(n_components=2)`, puis tracez un
nuage de points colore par cluster (ajoutez les centres si vous le souhaitez).

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(X_std)                 # TODO : fit_transform(X_std)
var = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(6.4, 5))
for c in sorted(np.unique(labels)):
    m = labels == c
    # TODO : scatter des points du cluster c (coords[m, 0], coords[m, 1])
    ax.scatter(coords[m, 0], coords[m, 1], label=f'Cluster {c}', alpha=0.7)
ax.set(title="Clusters (ACP)", xlabel=f"PC1 ({var[0]:.0%})", ylabel=f"PC2 ({var[1]:.0%})")
ax.legend(frameon=False)
plt.show()

## 5 & 6. Integration + qualification
**Consigne.** Ajoutez la colonne `cluster` a une copie de `X`, puis calculez le
profil moyen (`groupby`) des variables `alcohol`, `color_intensity`,
`flavanoids`, `proline` par cluster. Commentez les profils.

In [ ]:
vins = X.copy()
vins["cluster"] = labels
cles = ["alcohol", "color_intensity", "flavanoids", "proline"]
# TODO : profil moyen par cluster
profil =  vins.groupby('cluster')[cles].mean()
profil
 Qualification de chaque cluster :
 
Cluster 0 (Cépage 1) : 
Alcool modéré (~13.4)
Faible intensité colorée (~4.6) 
Très riche en flavanoïdes (~2.8),Très riche en proline (~1000)
 Vins élégants, fins, avec des tanins soyeux
Cluster 1 (Cépage 2) :
Alcool élevé (~14.0)
 Intensité colorée moyenne (~6.7),Flavanoïdes moyens (~2.3)
 Proline modérée (~800)
 Vins puissants, structurés, avec bonne matière
 Cluster 2 (Cépage 3) :
Alcool plus faible (~12.2)
Forte intensité colorée (~9.6), Faibles flavanoïdes (~1.0)
Très faible proline (~350),Vins colorés mais moins complexes, tanins plus durs

## A rendre
- Le `k` retenu et sa justification (coude + silhouette).
- L'ARI obtenu et votre interpretation du tableau croise.
- Une phrase de qualification par cluster.

**Bonus.** Refaites l'analyse **sans** standardisation : que devient l'ARI ?